# H3 polygon polyfill

Step-by-step BFS from a seed cell at each part's `representative_point()` (guaranteed on/in the polygon), expanding via `intersects(polygon)`, then a spatial predicate filter — same multipolygon style as [`02_a5_linetrace.ipynb`](../linetrace/02_a5_linetrace.ipynb).

**Seed:** `polygon.representative_point()` → `h3.latlng_to_cell` (Shapely also has `point_on_surface()`; both aim for an interior point, unlike the bbox centroid).

**Neighbor lookup:** `h3.grid_disk(cell, 1)` (6 edge-sharing hex neighbors around a cell; pentagons have 5).

**Note:** vgrid [`polygon2h3`](https://github.com/opengeoshub/vgrid/blob/main/vgrid/conversion/vector2dggs/vector2h3.py) uses `h3.geo_to_cells(bbox, resolution)` in one shot — see [`01_h3_polyfill_2.ipynb`](01_h3_polyfill_2.ipynb). Optional `COMPACT=True` animates upstream **`compactCells`** ([h3lib `h3Index.c`](https://github.com/uber/h3/blob/master/src/h3lib/lib/h3Index.c)) — one resolution level per round, full sibling sets merge to parent (7 hex / 6 pentagon children) — same logic as [`../compact/01_h3_compact.ipynb`](../compact/01_h3_compact.ipynb).

Input: [`multipolygon.geojson`](https://raw.githubusercontent.com/opengeoshub/vopendata/main/shape/multipolygon.geojson) — **12 features**, **13 polygon parts**. **H3 resolution** is shown on every frame.

## Install necessary packages

In [ ]:
# %pip install vgrid geopandas matplotlib imageio pillow
# optional for MP4:
%pip install imageio-ffmpeg

In [ ]:
"""Step-by-step polygon2h3-style BFS animation (multipolygon.geojson)."""
from collections import defaultdict, deque
from pathlib import Path

import geopandas as gpd
import h3
import imageio.v2 as imageio
import matplotlib.pyplot as plt
from matplotlib.collections import PatchCollection
from matplotlib.patches import Polygon as MplPolygon
from shapely.geometry import MultiPolygon, box

from vgrid.conversion.dggs2geo.h32geo import h32geo
from vgrid.conversion.vector2dggs.vector2h3 import polygon2h3
from vgrid.utils.geometry import check_predicate

URL = "https://raw.githubusercontent.com/opengeoshub/vopendata/main/shape/polygon2.geojson"
RESOLUTION = 10
PREDICATE = "within"
COMPACT = True
OUT_GIF = "polygon2h3.gif"
OUT_MP4 = "polygon2h3.mp4"
FRAME_EVERY_N_BFS = 8
FRAME_EVERY_MERGE = 1
DPI = 120
PART_COLORS = ["#1f4e79", "#c55a11", "#2e7d32", "#b71c1c", "#6a1b9a", "#4e342e", "#00695c", "#5d4037"]


def cell_patches(cell_polys, facecolor, edgecolor, alpha=0.55, lw=0.4):
    patches = []
    for poly in cell_polys:
        if poly is None or poly.is_empty:
            continue
        patches.append(MplPolygon(list(poly.exterior.coords), closed=True))
    return PatchCollection(
        patches, facecolor=facecolor, edgecolor=edgecolor, alpha=alpha, linewidths=lw
    )


def polygons_from_feature(feature):
    if feature.geom_type == "Polygon":
        return [feature]
    if feature.geom_type == "MultiPolygon":
        return list(feature.geoms)
    return []


def polygons_from_gdf(gdf):
    parts = []
    for geom in gdf.geometry:
        if geom is None or geom.is_empty:
            continue
        parts.extend(polygons_from_feature(geom))
    return parts


def feature_from_gdf(gdf):
    parts = polygons_from_gdf(gdf)
    if not parts:
        raise ValueError("No polygon geometries found in input GeoJSON")
    if len(parts) == 1:
        return parts[0]
    return MultiPolygon(parts)


def part_color(part_index):
    return PART_COLORS[part_index % len(PART_COLORS)]


def render_frame(
    parts,
    bbox,
    seed_poly,
    active_cells,
    title,
    path,
    resolution,
    current_poly=None,
    active_part=None,
):
    fig, ax = plt.subplots(figsize=(8, 8))
    union = MultiPolygon(parts) if len(parts) > 1 else parts[0]
    minx, miny, maxx, maxy = union.bounds
    pad = max(maxx - minx, maxy - miny) * 0.08 or 0.01
    ax.set_xlim(minx - pad, maxx + pad)
    ax.set_ylim(miny - pad, maxy + pad)

    for i, poly in enumerate(parts):
        color = part_color(i)
        lw = 3.0 if active_part == i else 1.8
        alpha = 1.0 if active_part is None or active_part == i else 0.45
        gpd.GeoSeries([poly]).plot(
            ax=ax, facecolor="none", edgecolor=color, lw=lw, alpha=alpha
        )
    if bbox is not None:
        gpd.GeoSeries([bbox]).plot(
            ax=ax, facecolor="none", edgecolor="#ff7f0e", lw=1.5, linestyle="--"
        )

    visited = list(active_cells) if active_cells else []
    if current_poly is not None:
        visited = [
            p
            for p in visited
            if p is not current_poly and not p.equals(current_poly)
        ]
    if visited:
        ax.add_collection(cell_patches(visited, "#2ca02c", "#1a5f1a", alpha=0.45))
    if seed_poly is not None and (
        current_poly is None
        or (seed_poly is not current_poly and not seed_poly.equals(current_poly))
    ):
        ax.add_collection(cell_patches([seed_poly], "#d62728", "#8b0000", alpha=0.7))
    if current_poly is not None:
        ax.add_collection(
            cell_patches([current_poly], "#ffcc00", "#cc8800", alpha=0.9, lw=2.5)
        )
    ax.plot([], [], color="#ffcc00", lw=4, label="current cell (deque)")
    ax.plot([], [], color="#2ca02c", lw=4, label="visited")
    ax.plot([], [], color="#d62728", lw=4, label="seed")
    ax.legend(loc="upper right", fontsize=8)
    ax.text(
        0.02,
        0.98,
        f"H3 resolution: {resolution}",
        transform=ax.transAxes,
        fontsize=9,
        va="top",
        ha="left",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.9),
        zorder=6,
    )
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.25)
    fig.subplots_adjust(left=0.08, right=0.92, top=0.92, bottom=0.08)
    fig.savefig(path, dpi=DPI, facecolor="white")
    plt.close(fig)


def render_compact_frame(
    parts,
    background_cells,
    title,
    path,
    resolution,
    child_polys=None,
    parent_poly=None,
):
    fig, ax = plt.subplots(figsize=(8, 8))
    union = MultiPolygon(parts) if len(parts) > 1 else parts[0]
    minx, miny, maxx, maxy = union.bounds
    pad = max(maxx - minx, maxy - miny) * 0.08 or 0.01
    ax.set_xlim(minx - pad, maxx + pad)
    ax.set_ylim(miny - pad, maxy + pad)

    for i, poly in enumerate(parts):
        gpd.GeoSeries([poly]).plot(
            ax=ax, facecolor="none", edgecolor=part_color(i), lw=1.8, alpha=0.7
        )

    if background_cells:
        ax.add_collection(cell_patches(background_cells, "#e8eaf6", "#5c6bc0", alpha=0.45))
    if child_polys:
        ax.add_collection(cell_patches(child_polys, "#00bcd4", "#006064", alpha=0.9, lw=1.4))
    if parent_poly is not None:
        ax.add_collection(
            cell_patches([parent_poly], "#ff9800", "#e65100", alpha=0.55, lw=2.0)
        )
        gpd.GeoSeries([parent_poly.boundary]).plot(
            ax=ax, color="#e65100", lw=3.5, linestyle="--", zorder=4
        )

    ax.plot([], [], color="#00bcd4", lw=4, label="children (merge group)")
    ax.plot([], [], color="#ff9800", lw=4, label="parent cell")
    ax.plot([], [], color="#5c6bc0", lw=4, label="other cells")
    ax.legend(loc="upper right", fontsize=8)
    ax.text(
        0.02,
        0.98,
        f"H3 resolution: {resolution}",
        transform=ax.transAxes,
        fontsize=9,
        va="top",
        ha="left",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.9),
        zorder=6,
    )
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.25)
    fig.subplots_adjust(left=0.08, right=0.92, top=0.92, bottom=0.08)
    fig.savefig(path, dpi=DPI, facecolor="white")
    plt.close(fig)


def child_count_label(parent_id):
    n = len(h3.cell_to_children(parent_id))
    kind = "pentagon" if h3.is_pentagon(parent_id) else "hex"
    return f"{n} {kind} children"


def compact_one_round(current_ids):
    """One outer-loop iteration of h3lib compactCells (h3Index.c).

    At resolution r, group by cell_to_parent(·, r-1). Parents with a full
    child set merge; other cells are finalized for output.
    """
    if not current_ids:
        return set(), set(), []

    res = h3.get_resolution(next(iter(current_ids)))
    if res == 0:
        return set(), set(current_ids), []

    parent_res = res - 1
    grouped = defaultdict(set)
    for cell_id in current_ids:
        grouped[h3.cell_to_parent(cell_id, parent_res)].add(cell_id)

    merges = []
    compactable = set()
    for parent, children in grouped.items():
        if children == set(h3.cell_to_children(parent)):
            compactable.add(parent)
            merges.append((parent, sorted(children)))

    merges.sort(key=lambda item: item[0])  # stable sweep order by parent H3 index

    remaining = set()
    finalized = set()
    for cell_id in current_ids:
        parent = h3.cell_to_parent(cell_id, parent_res)
        if parent in compactable:
            remaining.add(parent)
        else:
            finalized.add(cell_id)

    return remaining, finalized, merges


def validate_same_resolution(cell_ids):
    resolutions = {h3.get_resolution(c) for c in cell_ids}
    if len(resolutions) > 1:
        raise ValueError(
            f"Input cells must share the same resolution; got {sorted(resolutions)}"
        )


def polys_for_ids(cell_ids):
    polys = []
    for cell_id in cell_ids:
        poly = h32geo(cell_id)
        if poly is not None and not poly.is_empty:
            polys.append(poly)
    return polys


def h3_compact_with_frames(cell_ids, resolution, snap, snap_merge):
    """Step through h3lib compactCells rounds; verify with h3.compact_cells."""
    validate_same_resolution(cell_ids)
    current = set(cell_ids)
    finalized = set()
    id_to_poly = {}

    def poly_for(cell_id):
        if cell_id not in id_to_poly:
            id_to_poly[cell_id] = h32geo(cell_id)
        return id_to_poly[cell_id]

    snap(
        f"8. Before compact ({len(current)} cells, res {resolution})",
        active=polys_for_ids(sorted(current)),
    )

    round_idx = 0
    while current:
        res = h3.get_resolution(next(iter(current)))
        if res == 0:
            finalized.update(current)
            snap(
                f"9c. Resolution-0 cells finalized ({len(current)} cells)",
                active=polys_for_ids(sorted(finalized)),
            )
            break

        remaining, newly_finalized, merges = compact_one_round(current)
        if not merges:
            finalized.update(current)
            break

        round_idx += 1
        for merge_i, (parent_id, child_ids) in enumerate(merges, start=1):
            if merge_i % FRAME_EVERY_MERGE != 0 and merge_i != len(merges):
                continue
            child_set = set(child_ids)
            background = [
                poly_for(cid)
                for cid in current
                if cid not in child_set
                and poly_for(cid) is not None
                and not poly_for(cid).is_empty
            ]
            children = [
                poly_for(cid)
                for cid in child_ids
                if poly_for(cid) is not None and not poly_for(cid).is_empty
            ]
            snap_merge(
                f"9a. Round {round_idx} res {res}→{res - 1} merge {merge_i}/{len(merges)}: "
                f"{child_count_label(parent_id)} → {parent_id}",
                background,
                children,
                poly_for(parent_id),
            )

        finalized.update(newly_finalized)
        current = remaining
        snap(
            f"9b. After round {round_idx}: {len(finalized)} finalized, "
            f"{len(current)} still compacting",
            active=polys_for_ids(sorted(finalized | current)),
        )

    animated_ids = sorted(finalized | current)
    ref_ids = sorted(h3.compact_cells(cell_ids))
    snap(
        f"10. Final compact set ({len(ref_ids)} cells) — h3.compact_cells",
        active=polys_for_ids(ref_ids),
    )
    if set(animated_ids) != set(ref_ids):
        print(
            "Warning: compact animation differs from h3.compact_cells:",
            len(animated_ids),
            "vs",
            len(ref_ids),
        )
    else:
        print(f"Verified: compact animation matches h3.compact_cells ({len(ref_ids)} cells)")
    return ref_ids


def polygon2h3_with_frames(parts, feature, resolution, predicate, frame_dir, compact=False):
    frame_dir.mkdir(parents=True, exist_ok=True)
    frames = []
    idx = 0
    merged_ids = []
    merged_polys = []
    seen_ids = set()
    accumulated = []

    def snap(title, bbox=None, seed=None, active=None, current=None, active_part=None):
        nonlocal idx
        p = frame_dir / f"frame_{idx:04d}.png"
        render_frame(
            parts,
            bbox,
            seed,
            active if active is not None else accumulated,
            title,
            p,
            resolution,
            current_poly=current,
            active_part=active_part,
        )
        frames.append(p)
        idx += 1

    n_parts = len(parts)
    snap(
        f"1. Input res {resolution} ({n_parts} polygon part{'s' if n_parts != 1 else ''})"
    )
    snap("2. All parts (distinct colors)")

    for part_i, polygon in enumerate(parts, start=1):
        min_lng, min_lat, max_lng, max_lat = polygon.bounds
        bbox = box(min_lng, min_lat, max_lng, max_lat)
        seed_pt = polygon.representative_point()  # guaranteed on/in polygon (not bbox centre)
        lon, lat = seed_pt.x, seed_pt.y
        seed_id = h3.latlng_to_cell(lat, lon, resolution)
        seed_poly = h32geo(seed_id)
        part_idx = part_i - 1

        snap(
            f"3. Part {part_i}/{n_parts}: seed at representative_point",
            bbox=bbox,
            seed=seed_poly,
            active_part=part_idx,
        )

        neighbor_polys = []
        for nid in h3.grid_disk(seed_id, 1):
            if nid == seed_id:
                continue
            npoly = h32geo(nid)
            if npoly is not None and not npoly.is_empty:
                neighbor_polys.append(npoly)
        snap(
            f"3b. Part {part_i} seed neighbors (grid_disk, {len(neighbor_polys)} cells)",
            bbox=bbox,
            seed=seed_poly,
            active=neighbor_polys,
            active_part=part_idx,
        )

        if seed_poly.contains(polygon):
            if seed_id not in seen_ids:
                seen_ids.add(seed_id)
                merged_ids.append(seed_id)
                merged_polys.append(seed_poly)
                accumulated.append(seed_poly)
            snap(
                f"4. Part {part_i} seed covers polygon — single cell",
                bbox=bbox,
                seed=seed_poly,
                active=[seed_poly],
                active_part=part_idx,
            )
            continue

        intersecting = {}
        covered = set()
        queue = deque([seed_id])
        bfs_step = 0
        while queue:
            cid = queue.popleft()
            if cid in covered:
                continue
            covered.add(cid)
            cell_poly = h32geo(cid)
            if cell_poly is None or cell_poly.is_empty:
                continue
            if cell_poly.intersects(polygon):
                intersecting[cid] = cell_poly
                for nid in h3.grid_disk(cid, 1):
                    if nid not in covered:
                        queue.append(nid)
                bfs_step += 1
                if bfs_step % FRAME_EVERY_N_BFS == 0:
                    snap(
                        f"4. Part {part_i} BFS step {bfs_step}: {cid} ({len(intersecting)} polygon hits)",
                        bbox=bbox,
                        seed=seed_poly,
                        active=list(intersecting.values()),
                        current=cell_poly,
                        active_part=part_idx,
                    )
        snap(
            f"5. Part {part_i} BFS complete ({len(intersecting)} polygon hits)",
            bbox=bbox,
            seed=seed_poly,
            active=list(intersecting.values()),
            active_part=part_idx,
        )

        part_final = []
        for cid, cell_poly in intersecting.items():
            if check_predicate(cell_poly, polygon, predicate):
                part_final.append((cid, cell_poly))
                if cid not in seen_ids:
                    seen_ids.add(cid)
                    merged_ids.append(cid)
                    merged_polys.append(cell_poly)
                    accumulated.append(cell_poly)
        snap(
            f"6. Part {part_i} after predicate '{predicate}' ({len(part_final)} cells)",
            bbox=bbox,
            seed=seed_poly,
            active=[p for _, p in part_final],
            active_part=part_idx,
        )

    snap(
        f"7. Merged result ({len(merged_ids)} cells across {n_parts} parts)",
        active=merged_polys,
    )

    final_ids = merged_ids

    if compact and merged_ids:

        def snap_merge(title, background, children, parent_poly):
            nonlocal idx
            p = frame_dir / f"frame_{idx:04d}.png"
            render_compact_frame(
                parts,
                background,
                title,
                p,
                resolution,
                child_polys=children,
                parent_poly=parent_poly,
            )
            frames.append(p)
            idx += 1

        final_ids = h3_compact_with_frames(
            merged_ids, resolution, snap, snap_merge
        )

    from_poly = []
    for part in parts:
        rows = polygon2h3(
            part, resolution, predicate=predicate, compact=compact
        )
        from_poly.extend(row["h3"] for row in rows)
    ref_ids = set(from_poly)
    our_ids = set(final_ids)
    if our_ids != ref_ids:
        print(
            "Warning: cell set differs from polygon2h3:",
            len(our_ids),
            "vs",
            len(ref_ids),
        )

    return frames, final_ids


def main():
    gdf = gpd.read_file(URL)
    parts = polygons_from_gdf(gdf)
    feature = feature_from_gdf(gdf)
    print(f"Loaded {len(gdf)} feature(s), {len(parts)} polygon part(s)")

    frame_dir = Path("_polygon2h3_frames")
    frames, cell_ids = polygon2h3_with_frames(
        parts, feature, RESOLUTION, PREDICATE, frame_dir, compact=COMPACT
    )
    imageio.mimsave(OUT_GIF, [imageio.imread(f) for f in frames], duration=0.9)
    compact_note = f" (compact)" if COMPACT else ""
    print(
        f"Wrote {OUT_GIF} ({len(frames)} frames, {len(cell_ids)} final cells{compact_note})"
    )
    try:
        writer = imageio.get_writer(OUT_MP4, fps=1.2)
        for f in frames:
            writer.append_data(imageio.imread(f))
        writer.close()
        print(f"Wrote {OUT_MP4}")
    except Exception as e:
        print(f"MP4 skipped ({e}). GIF is enough.")


if __name__ == "__main__":
    main()
